# Ejercicio 2 — Transformaciones de Variables Normales
## Máster Executive en Finanzas Cuantitativas 2026 — AFI Global Education
### Fundamentos Matemáticos: Probabilidad y Simulación

---

**Objetivo:** Sea $X \sim \mathcal{N}(0,1)$. Estudiar las distribuciones de las variables transformadas:

$$g(x) = \begin{cases} -x & \text{si } x < 0 \\ \alpha x & \text{si } x \geq 0 \end{cases}, \qquad h(x) = \begin{cases} -x & \text{si } x < 0 \\ \sqrt{x} & \text{si } x \geq 0 \end{cases}$$

Para $Y = g(X)$ se estudia el efecto del parámetro $\alpha$ y se deriva analíticamente $f_Y$. Para $Z = h(X)$ se demuestra y verifica la densidad $f_Z$.

**Puntuación:** 2 puntos (3 apartados de igual peso).

---
## Configuración del entorno

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats, integrate
import os

# ── Estilo gráfico coherente ────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# ── Semilla fija (reproducibilidad) ────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── Constantes del problema ─────────────────────────────────────────────────
N_SIM = 100_000          # muestras para simulación
ALPHAS = [-1, -0.5, 0, 0.5, 1]  # valores de α del enunciado

# ── Directorio de salida ────────────────────────────────────────────────────
os.makedirs("resultados", exist_ok=True)
print("Entorno configurado. Seed =", SEED, "| N_SIM =", N_SIM)

In [ ]:
# ── Simular la base: X ~ N(0,1) ────────────────────────────────────────────
X = rng.standard_normal(N_SIM)
print(f"X ~ N(0,1): media={X.mean():.4f}, std={X.std():.4f}  (esperado: 0, 1)")

---
## Apartado 1 — Densidad de $Y = g(X)$ por simulación para distintos $\alpha$

### Definición de la transformación

$$g(x) = \begin{cases} -x & \text{si } x < 0 \\ \alpha x & \text{si } x \geq 0 \end{cases}$$

Para $x < 0$: $g(x) = -x > 0$ (refleja la parte negativa al eje positivo).  
Para $x \geq 0$: $g(x) = \alpha x$ (escala la parte positiva por $\alpha$).

### Análisis cualitativo por casos

| $\alpha$ | Comportamiento | Soporte de $Y$ | Continuidad |
|---|---|---|---|
| $-1$ | $g(x) = -x$ para todo $x$ → $Y = -X \sim \mathcal{N}(0,1)$ | $\mathbb{R}$ | Continua (normal estándar) |
| $-0.5$ | parte $\geq 0$ va a $(-\infty, 0]$ | $\mathbb{R}$ | Continua pero asimétrica |
| $0$ | $g(x)=0$ para $x\geq 0$: **masa puntual en 0** | $\{0\}\cup(0,\infty)$ | **Mixta** (no continua) |
| $0.5$ | ambas ramas positivas, misma forma | $(0,\infty)$ | Continua |
| $1$ | $g(x)=|x|$ → $Y=|X|$ = **semi-normal** | $[0,\infty)$ | Continua (half-normal) |

$Y$ es continua si y sólo si $\alpha \neq 0$.

In [ ]:
# ── Función de transformación g ────────────────────────────────────────────
def transform_g(x, alpha):
    """Y = g(X): -x si x<0, alpha*x si x>=0."""
    return np.where(x < 0, -x, alpha * x)

# ── Simular Y para cada alpha ───────────────────────────────────────────────
Y_samples = {alpha: transform_g(X, alpha) for alpha in ALPHAS}

# Verificar estadísticos básicos
print(f"{'Alpha':>6}  {'Media':>8}  {'Std':>8}  {'Min':>8}  {'Max':>8}")
print("-" * 48)
for a, ys in Y_samples.items():
    print(f"{a:>6}  {ys.mean():>8.4f}  {ys.std():>8.4f}  {ys.min():>8.4f}  {ys.max():>8.4f}")

In [ ]:
# ── Figura: histogramas de Y para cada alpha + N(0,1) de referencia ─────────
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=False)

x_ref = np.linspace(-4, 4, 400)
phi   = stats.norm.pdf(x_ref)   # N(0,1) de referencia

for ax, alpha in zip(axes, ALPHAS):
    ys = Y_samples[alpha]
    # Ajustar rango del histograma
    lo, hi = np.percentile(ys, [0.5, 99.5])
    ax.hist(ys, bins=80, density=True, alpha=0.55, color="steelblue",
            range=(lo, hi), label="Simulación")
    # Superponer N(0,1) como referencia visual
    x_r = np.linspace(lo, hi, 300)
    ax.plot(x_r, stats.norm.pdf(x_r), 'r--', lw=1.5, label=r"$\mathcal{N}(0,1)$")
    ax.set_title(f"$\\alpha = {alpha}$", fontsize=12)
    ax.set_xlabel("y")
    if ax == axes[0]:
        ax.set_ylabel("Densidad")
    ax.legend(fontsize=8)

fig.suptitle(r"Densidad simulada de $Y = g(X)$ para distintos $\alpha$ — $X\sim\mathcal{N}(0,1)$",
             fontsize=13)
fig.tight_layout()
fig.savefig("resultados/grafico_04_fY_por_alpha.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en resultados/")

### Discusión: continuidad y casos especiales

- **$\alpha = -1$:** $g(x) = -x$ para todo $x$, por tanto $Y = -X \sim \mathcal{N}(0,1)$. La distribución es idéntica a la normal estándar.
- **$\alpha = 0$:** Para $x \geq 0$ (que ocurre con probabilidad 1/2), $Y = 0$. Hay una **masa puntual** de probabilidad $P(Y=0) = 1/2$ en el origen. $Y$ **no es continua**: es una variable mixta.
- **$\alpha = 1$:** $g(x) = |x|$, por tanto $Y = |X|$ sigue la distribución **semi-normal** (half-normal). Su densidad es el doble de la normal solo para $y \geq 0$.
- **$\alpha \in \{-0.5, 0.5\}$:** $Y$ es continua pero no normal. La asimetría depende de si $\alpha > 1$ o $\alpha < 1$ (escala diferente en parte positiva vs negativa).

---
## Apartado 2 — Derivación analítica de $f_Y(y)$ para $\alpha > 0$

### Método: función de distribución acumulada (CDF)

Para $\alpha > 0$, la transformación $g$ mapea:
- $x < 0 \Rightarrow y = -x > 0$ (parte negativa, refleja al positivo)
- $x \geq 0 \Rightarrow y = \alpha x \geq 0$ (parte positiva, escala)

Ambas ramas producen valores en $[0, \infty)$, por tanto **$Y$ solo toma valores positivos**.

Para $y > 0$, calculamos $F_Y(y) = P(Y \leq y) = P(g(X) \leq y)$:

$$P(Y \leq y) = P\bigl(\{x < 0,\, -x \leq y\}\bigr) + P\bigl(\{x \geq 0,\, \alpha x \leq y\}\bigr)$$

$$= P(-y \leq X < 0) + P\!\left(0 \leq X \leq \frac{y}{\alpha}\right)$$

$$= P(X \leq 0) - P(X \leq -y) + P\!\left(X \leq \frac{y}{\alpha}\right) - P(X \leq 0)$$

$$= \Phi(0) - \Phi(-y) + \Phi\!\left(\frac{y}{\alpha}\right) - \Phi(0)$$

$$\boxed{F_Y(y) = \Phi\!\left(\frac{y}{\alpha}\right) - \Phi(-y)}, \quad y > 0.$$

Derivando respecto a $y$:

$$\boxed{f_Y(y) = \frac{1}{\alpha}\phi\!\left(\frac{y}{\alpha}\right) + \phi(y)}, \quad y > 0,$$

donde $\phi(t) = \frac{1}{\sqrt{2\pi}}e^{-t^2/2}$ es la densidad normal estándar.

**Interpretación:** La densidad de $Y$ es la suma de dos contribuciones:
- $\phi(y)$: contribución de $X < 0$ (reflejada): probabilidad de que $-X = y$
- $\frac{1}{\alpha}\phi(y/\alpha)$: contribución de $X \geq 0$ escalada: probabilidad de que $\alpha X = y$

**Verificación:** $\int_0^\infty f_Y(y)\,dy = \int_0^\infty \frac{1}{\alpha}\phi(y/\alpha)\,dy + \int_0^\infty \phi(y)\,dy = \frac{1}{2} + \frac{1}{2} = 1$ ✓

In [ ]:
# ── Densidad analítica de Y para alpha > 0 ─────────────────────────────────
def f_Y_analitica(y, alpha):
    """
    f_Y(y) = (1/alpha)*phi(y/alpha) + phi(y),  y > 0, alpha > 0
    donde phi es la densidad N(0,1)
    """
    assert alpha > 0, "Esta fórmula es válida solo para alpha > 0"
    return (1/alpha) * stats.norm.pdf(y/alpha) + stats.norm.pdf(y)

# ── Verificación: normalización ─────────────────────────────────────────────
for alpha in [0.5, 1, 2]:
    integral, _ = integrate.quad(lambda y: f_Y_analitica(y, alpha), 0, np.inf)
    print(f"alpha={alpha}: ∫₀^∞ f_Y(y)dy = {integral:.8f}  (debe ser 1.0) ✓")

In [ ]:
# ── Figura: densidad analítica vs simulación para alpha > 0 ────────────────
alphas_pos = [0.5, 1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, alpha in zip(axes, alphas_pos):
    ys = Y_samples[alpha]
    # Histograma simulado (solo parte positiva)
    ys_pos = ys[ys > 0]
    ax.hist(ys_pos, bins=80, density=True, alpha=0.5, color="steelblue",
            label=f"Simulación (N={N_SIM:,})")
    # Curva analítica
    y_plot = np.linspace(0.001, 4, 400)
    ax.plot(y_plot, f_Y_analitica(y_plot, alpha), 'r-', lw=2.5,
            label=r"$f_Y(y)=\frac{1}{\alpha}\phi\left(\frac{y}{\alpha}\right)+\phi(y)$")
    ax.set_title(f"$\\alpha = {alpha}$", fontsize=12)
    ax.set_xlabel("y")
    ax.set_ylabel("Densidad")
    ax.legend(fontsize=9)

fig.suptitle(r"$f_Y(y)$ analítica vs simulada — $\alpha > 0$", fontsize=13)
fig.tight_layout()
fig.savefig("resultados/grafico_05_fY_analitica_vs_sim.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en resultados/")

---
## Apartado 3 — Densidad de $Z = h(X)$

### Demostración analítica

La transformación $h$ es:

$$h(x) = \begin{cases} -x & \text{si } x < 0 \\ \sqrt{x} & \text{si } x \geq 0 \end{cases}$$

Para $z > 0$, calculamos $F_Z(z) = P(Z \leq z)$:

$$P(Z \leq z) = P\bigl(\{x<0,\,-x\leq z\}\bigr) + P\bigl(\{x\geq 0,\,\sqrt{x}\leq z\}\bigr)$$

$$= P(-z \leq X < 0) + P(0 \leq X \leq z^2)$$

$$= \Phi(0) - \Phi(-z) + \Phi(z^2) - \Phi(0)$$

$$= \Phi(z^2) - \Phi(-z)$$

Derivando respecto a $z$:

$$f_Z(z) = \frac{d}{dz}\Phi(z^2) - \frac{d}{dz}\Phi(-z) = \phi(z^2)\cdot 2z + \phi(-z)\cdot 1$$

Como $\phi(-z) = \phi(z)$ (densidad normal es par):

$$\boxed{f_Z(z) = \frac{1}{\sqrt{2\pi}}\left(2z\,e^{-z^4/2} + e^{-z^2/2}\right)}, \quad z > 0.$$

que es exactamente lo pedido en el enunciado. $\blacksquare$

**Verificación de normalización:**

$$\int_0^\infty f_Z(z)\,dz = \int_0^\infty \frac{2z}{\sqrt{2\pi}}e^{-z^4/2}\,dz + \int_0^\infty \phi(z)\,dz$$

Para la primera integral, con $u = z^2$, $du = 2z\,dz$:

$$\int_0^\infty \frac{2z}{\sqrt{2\pi}}e^{-z^4/2}\,dz = \frac{1}{\sqrt{2\pi}}\int_0^\infty e^{-u^2/2}\,du = \frac{1}{\sqrt{2\pi}}\cdot\frac{\sqrt{2\pi}}{2} = \frac{1}{2}$$

La segunda integral es $\int_0^\infty \phi(z)\,dz = 1/2$. Total: $1/2 + 1/2 = 1$ ✓

In [ ]:
# ── Transformación h y densidad analítica de Z ──────────────────────────────
def transform_h(x):
    """Z = h(X): -x si x<0, sqrt(x) si x>=0."""
    return np.where(x < 0, -x, np.sqrt(np.maximum(x, 0)))

def f_Z_analitica(z):
    """f_Z(z) = (1/sqrt(2pi)) * (2z*exp(-z^4/2) + exp(-z^2/2)),  z > 0"""
    return (1/np.sqrt(2*np.pi)) * (2*z*np.exp(-z**4/2) + np.exp(-z**2/2))

# ── Simular Z ──────────────────────────────────────────────────────────────
Z = transform_h(X)
print(f"Z: media={Z.mean():.4f}, std={Z.std():.4f}")
print(f"Soporte: min={Z.min():.4f}, max={Z.max():.4f}")

# ── Verificación: normalización de f_Z ──────────────────────────────────────
integral_Z, _ = integrate.quad(f_Z_analitica, 0, np.inf)
print(f"\n∫₀^∞ f_Z(z) dz = {integral_Z:.8f}  (debe ser 1.0)")
assert abs(integral_Z - 1.0) < 1e-6, "f_Z no integra a 1"
print("✓ f_Z es una densidad válida")

In [ ]:
# ── Test Kolmogorov-Smirnov: simulación vs densidad analítica ──────────────
# Para el KS necesitamos la CDF: F_Z(z) = Phi(z²) - Phi(-z)
def F_Z_analitica(z):
    return stats.norm.cdf(z**2) - stats.norm.cdf(-z)

# KS test (muestral vs teórico)
ks_stat, ks_pval = stats.kstest(Z, F_Z_analitica)
print(f"Test Kolmogórov-Smirnov:")
print(f"  Estadístico D = {ks_stat:.5f}")
print(f"  p-valor       = {ks_pval:.4f}")
if ks_pval > 0.05:
    print("  ✓ No se rechaza que Z sigue la densidad teórica (p > 0.05)")
else:
    print("  ✗ Se rechaza la hipótesis nula")

In [ ]:
# ── Figura: f_Z simulada vs analítica ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel izquierdo: histograma vs densidad teórica
ax = axes[0]
ax.hist(Z, bins=100, density=True, alpha=0.5, color="steelblue",
        label=f"Simulación (N={N_SIM:,})", range=(0, 4))
z_plot = np.linspace(0.001, 4, 500)
ax.plot(z_plot, f_Z_analitica(z_plot), 'r-', lw=2.5,
        label=r"$f_Z(z)=\frac{1}{\sqrt{2\pi}}(2ze^{-z^4/2}+e^{-z^2/2})$")
ax.set_xlabel("z")
ax.set_ylabel("Densidad")
ax.set_title(r"$f_Z(z)$: simulación vs densidad analítica")
ax.legend(fontsize=9)

# Panel derecho: diferencia absoluta histograma-densidad
ax2 = axes[1]
counts, edges = np.histogram(Z, bins=100, density=True, range=(0, 4))
mids = (edges[:-1] + edges[1:]) / 2
teorico = f_Z_analitica(mids)
ax2.bar(mids, np.abs(counts - teorico), width=edges[1]-edges[0],
        color="orange", alpha=0.7, label="|Empírico - Teórico|")
ax2.set_xlabel("z")
ax2.set_ylabel("Diferencia absoluta")
ax2.set_title("Diferencia |empírico − teórico|")
ax2.legend()
print(f"Diferencia máxima: {np.abs(counts-teorico).max():.5f}")

fig.suptitle(r"Ejercicio 2.3 — Verificación de $f_Z(z)$ por simulación", fontsize=13)
fig.tight_layout()
fig.savefig("resultados/grafico_06_fZ_verificacion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada en resultados/")

---
## Conclusiones

In [ ]:
from IPython.display import display, HTML

tabla = [
    ("Concepto", "Resultado"),
    ("Y continua", "α ≠ 0"),
    ("α = 0: tipo distribución", "Mixta: masa en 0 con prob 1/2, continua en (0,∞)"),
    ("α = 1: Y = |X|", "Half-normal: f_Y(y) = 2φ(y), y > 0"),
    ("α = -1: Y = -X", "Normal estándar N(0,1)"),
    ("f_Y(y) para α > 0", "(1/α)φ(y/α) + φ(y),  y > 0"),
    ("f_Z(z)", "(1/√2π)(2z·e^{-z⁴/2} + e^{-z²/2}),  z > 0"),
    ("KS p-valor (f_Z)", f"{ks_pval:.4f} → no se rechaza la densidad teórica"),
    ("Normalización f_Z", f"{integral_Z:.8f} ≈ 1 ✓"),
]

html = "<table border='1' style='border-collapse:collapse;font-size:13px;'>"
for i, (k, v) in enumerate(tabla):
    bg = "#f2f2f2" if i % 2 == 0 else "white"
    bold = " font-weight:bold;" if i == 0 else ""
    html += f"<tr style='background:{bg};{bold}'><td style='padding:6px 12px'>{k}</td><td style='padding:6px 12px'>{v}</td></tr>"
html += "</table>"
display(HTML(html))

print("\nEjercicio 2 completado. Todos los resultados verificados analítica y numéricamente.")